# Домашнее задание к семинару 13 (HW13)
Тема: токенизация текста, инференс готовой BERT-подобной модели и базовый fine-tuning для классификации текста.

## Импорты, seed и среда

In [ ]:
# Базовые библиотеки для воспроизводимости, анализа и визуализации.
import random
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import datasets
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 4)

In [ ]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

## Данные и первичный анализ

In [ ]:
df = datasets.load_dataset("emotion")

train_df = pd.DataFrame({
    "text": df["train"]["text"],
    "label": df["train"]["label"]})

val_df = pd.DataFrame({
    "text": df["validation"]["text"],
    "label": df["validation"]["label"]})

test_df = pd.DataFrame({
    "text": df["test"]["text"],
    "label": df["test"]["label"]})

In [ ]:
# Смотрим размер датасета и распределение по классам.
print("Размер train выборки:", len(train_df))
print("Размер val выборки:", len(val_df))
print("Размер test выборки:", len(test_df))

display(train_df["label"].value_counts())

display(train_df.sample(6, random_state=42).reset_index(drop=True))

In [ ]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nРаспределение классов в train:")
display(train_df["label"].value_counts())

print("Распределение классов в validation:")
display(val_df["label"].value_counts())

print("Распределение классов в test:")
display(test_df["label"].value_counts())

In [ ]:
train_df.head(10)

Классифицируются сообщения в Twitter выражающие шесть базовых эмоций: гнев, страх, радость, любовь, грусть, удивление. Колонки: text, label.

## Токенизация

In [ ]:
# sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5)
label_names = df["train"].features["label"].names

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

for df in (train_df, val_df, test_df):
    df["label_name"] = df["label"].map(id2label)

    print("label2id:", label2id)
    print("id2label:", id2label)

In [ ]:
# Приводим датафреймы к формату HuggingFace Dataset.
train_ds = Dataset.from_pandas(
    train_df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True),
    preserve_index=False,
)
val_ds = Dataset.from_pandas(
    val_df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True),
    preserve_index=False,
)
test_ds = Dataset.from_pandas(
    test_df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True),
    preserve_index=False,
)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds,
})

dataset_dict

In [ ]:
# Посмотрим на несколько примеров.
print("Пример из train:")
display(dataset_dict["train"][:3])

print("Пример из validation:")
display(dataset_dict["validation"][:3])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Model checkpoint:", MODEL_NAME)

def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

# Удаляем столбец "text": DataCollatorWithPadding попытается преобразовать
# все поля датасета в тензоры, а строки в тензор не конвертируются.
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets

In [ ]:
# Data collator будет добавлять padding динамически, прямо при формировании батча.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_batch = [tokenized_datasets["train"][i] for i in range(3)]
collated_batch = data_collator(sample_batch)

for key, value in collated_batch.items():
    print(f"{key}: shape={tuple(value.shape)}")

In [ ]:
# Базовый разбор токенизации
tokenizer_example = AutoTokenizer.from_pretrained(MODEL_NAME)

def inspect_texts(texts: list, tokenizer) -> list:
    """Анализирует каждый текст отдельно и возвращает список DataFrame"""
    results = []

    for i, text in enumerate(texts):
        tokens = tokenizer.tokenize(text)
        token_ids = tokenizer.encode(text, add_special_tokens=True)

        df = pd.DataFrame({
            "position": list(range(len(tokens))),
            "token": tokens,
            "token_id": token_ids,
        })
        results.append(df)

        print(f"\n--- Текст {i+1} ---")
        print(f"Оригинал: {text}")
        print(f"Длина в токенах: {len(tokens)}")
        print(f"Специальные токены: [CLS] в начале, [SEP] в конце")
        print(df.to_string(index=False))
        print()

    return results

example_texts = train_ds[:3]["text"]

print("Исходные тексты:")
for i, text in enumerate(example_texts, 1):
    print(f"{i}. {text}")
print()

tokenized = tokenizer_example(
    example_texts,
    add_special_tokens=True,
    padding=True,
    return_tensors='pt'
)

print("\n1. Input IDs (с паддингом и спецтокенами):")
print(tokenized['input_ids'])
print(f"\nФорма: {tokenized['input_ids'].shape}")

print("\n2. Attention Mask:")
print(tokenized['attention_mask'])

print("\n3. Специальные токены:")
print(f"  [CLS] - {tokenizer_example.cls_token} (id: {tokenizer_example.cls_token_id}) - начало последовательности")
print(f"  [SEP] - {tokenizer_example.sep_token} (id: {tokenizer_example.sep_token_id}) - разделитель/конец")
print(f"  [PAD] - {tokenizer_example.pad_token} (id: {tokenizer_example.pad_token_id}) - паддинг")

print("\n4. Детальный разбор каждого текста:")
print("-" * 70)

# Анализируем каждый текст отдельно
for i, text in enumerate(example_texts):
    tokens = tokenizer_example.tokenize(text)
    token_ids = tokenizer_example.encode(text, add_special_tokens=True)

    print(f"\nТекст {i+1}: \"{text[:50]}{'...' if len(text) > 50 else ''}\"")
    print(f"   Длина: {len(tokens)} токенов")

    # Показываем токены с аннотацией спецтокенов
    print(f"\n   Токены и их ID:")
    for pos, (tok, tid) in enumerate(zip(tokens, token_ids)):
        if tid == tokenizer_example.cls_token_id:
            marker = " [CLS]"
        elif tid == tokenizer_example.sep_token_id:
            marker = " [SEP]"
        elif tid == tokenizer_example.pad_token_id:
            marker = " [PAD]"
        else:
            marker = ""
        print(f"      {pos:3d}: {tok:15s} -> {tid:5d}{marker}")

print("\n" + "=" * 70)
print("5. Декодирование западдингованных ID:")
decoded_batch = tokenizer_example.decode(tokenized['input_ids'][0])
print(f"Текст 1 (декодирован): {decoded_batch}")

# Показываем разницу между текстами разной длины
print("\n6. Демонстрация паддинга:")
lengths = [len(tokenizer_example.encode(t, add_special_tokens=True)) for t in example_texts]
max_len = max(lengths)
print(f"   Длины текстов в токенах: {lengths}")
print(f"   Максимальная длина: {max_len}")
print(f"   Короткие тексты дополнены до {max_len} токенами с помощью [PAD]")

print('\n' + '-' * 70)

## Инференс готовой модели

In [ ]:
# Загружаем модель для классификации.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:
demo_df = val_df.sample(5, random_state=42).reset_index(drop=True)
demo_texts = demo_df["text"].tolist()

demo_df

In [ ]:
pipeline_device = 0 if torch.cuda.is_available() else -1

text_clf = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=pipeline_device,
)

pipeline_outputs = text_clf(demo_texts)

pipeline_df = pd.DataFrame(
    {
        "text": demo_texts,
        "predicted_label": [item["label"] for item in pipeline_outputs],
        "score": [item["score"] for item in pipeline_outputs],
    }
)

pipeline_df

Данная модель обучена на sentiment analysis (рейтинги), а не на эмоциях, поэтому её предсказания не полностью соответствуют задаче emotion classification. Однако это также может быть  и ее преимуществом, так как в общем виде эмоции можно представить как рейтинг от негативного до позитивного.

## Fine-tuning для классификации текста

In [ ]:
# Загружаем модель для классификации.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:
# Функция метрик для Trainer.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

In [ ]:
# Общие параметры обучения.
common_training_kwargs = dict(
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=2,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

try:
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )
except TypeError:
    training_args = TrainingArguments(
        eval_strategy="epoch",
        save_strategy="epoch",
        **common_training_kwargs,
    )

training_args

In [ ]:
# Собираем Trainer и запускаем обучение.
# В transformers >= 5.0 аргумент tokenizer переименован в processing_class.
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

train_result = trainer.train()
train_result

In [ ]:
# История логов Trainer.
history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.head(10))

plt.figure(figsize=(8, 4))

if "loss" in history_df.columns:
    train_logs = history_df.dropna(subset=["loss"])
    plt.plot(train_logs["step"], train_logs["loss"], marker="o", label="train loss")

if "eval_loss" in history_df.columns:
    eval_logs = history_df.dropna(subset=["eval_loss"])
    plt.plot(eval_logs["step"], eval_logs["eval_loss"], marker="s", label="eval loss")

plt.title("История обучения")
plt.xlabel("Шаг")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Оценка качества и краткий анализ ошибок

In [ ]:
# В transformers >= 5.x NotebookProgressCallback теряет состояние после обучения,
# что вызывает RuntimeError при вызове evaluate() вне тренировочного цикла.
# Удаляем его перед standalone-оценкой – это стандартный обходной путь.
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

# Оценка Trainer на validation и test.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (int, float)) else f"{k}: {v}")

In [ ]:
# Детальные предсказания на тестовой выборке.
test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_true = test_output.label_ids

print("Classification report on test:")
print(
    classification_report(
        test_true,
        test_preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        zero_division=0,
    )
)

cm = confusion_matrix(test_true, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=30, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("./artifacts/confusion_matrix.png")
plt.show()

In [ ]:
# Таблица ошибок на тестовой выборке.
test_texts = test_df.reset_index(drop=True)["text"]
error_rows = []

for text, true_id, pred_id, prob_vector in zip(test_texts, test_true, test_preds, torch.softmax(torch.tensor(test_logits), dim=-1).numpy()):
    if true_id != pred_id:
        error_rows.append({
            "text": text,
            "true_label": id2label[int(true_id)],
            "pred_label": id2label[int(pred_id)],
            "pred_confidence": float(prob_vector[pred_id]),
        })

errors_df = pd.DataFrame(error_rows)

if len(errors_df) == 0:
    print("На тестовой выборке ошибок не найдено. Это возможно на маленьком учебном датасете.")
else:
    display(errors_df.sort_values(by="pred_confidence", ascending=False).reset_index(drop=True))

Из примеров ошибок можно сделать вывод, что модель часто не может распознать радость (joy) и печаль (sadness). А также очень часто путает похожие эмоции - радость и любовь.

In [ ]:
sample_df = pd.DataFrame({
    "text": test_texts,
    "true_label": [id2label[i] for i in test_true],
    "pred_label": [id2label[i] for i in test_preds],
    "confidence": np.max(torch.softmax(torch.tensor(test_logits), dim=-1).numpy(), axis=1),
})

sample_df.to_csv("./artifacts/sample_predictions.csv", index=False)